# Experimentation with GED lexicon matcher subsystem

- **Author**: Amir Anwar


In [1]:
import json
import os
import sys

import marisa_trie

# NOTE: import from root of the project
sys.path.append(os.path.abspath("../../../../"))

from src.services.ged.config import load_ged_config
from src.services.ged.detectors.lexicon import LexiconDetector
from src.services.ged.detectors.lexicon.trie_store import LexiconTrieStore
from src.services.ged.detectors.rule_based import RuleBasedDetector
from src.services.ged.orchestrator import GEDService
from src.services.ged.schemas import GEDInput
from src.services.preprocessing.orchestrator import preprocess
from src.services.preprocessing.schemas import PreprocessingInput

2026-06-15 21:34:02.385 | DEBUG    | src.services.ged.detectors.rule_based.registry:register_entry:99 - Registered entry: OT_ALIF_MAQSURA_PREP
2026-06-15 21:34:02.386 | DEBUG    | src.services.ged.detectors.rule_based.registry:register_entry:99 - Registered entry: OT_TA_MARBUTA_NOUN
2026-06-15 21:34:02.389 | WARNING  | src.services.ged.detectors.rule_based.loader:load_yaml_rules:193 - YAML file /home/amir/dev/baligh/src/services/ged/detectors/rule_based/rules/punctuation.yaml does not contain a rule list , skipping.
2026-06-15 21:34:02.389 | INFO     | src.services.ged.detectors.rule_based.loader:load_yaml_rules:230 - Loaded 2 YAML rules from /home/amir/dev/baligh/src/services/ged/detectors/rule_based/rules.
2026-06-15 21:34:02.390 | DEBUG    | src.services.ged.detectors.rule_based.detector:<module>:47 - Loaded 2 YAML rules from /home/amir/dev/baligh/src/services/ged/detectors/rule_based/rules
2026-06-15 21:34:02.392 | DEBUG    | src.services.ged.detectors.rule_based.registry:decorator

In [2]:
config = load_ged_config()
lexicon_config = config.lexicon

In [3]:
def make_store(
    *,
    words: list[str] | None = None,
    entity_phrases: list[str] | None = None,
    entity_tokens: list[str] | None = None,
) -> LexiconTrieStore:
    """Build a tiny trie store for deterministic notebook tests."""
    phrases = entity_phrases or []
    return LexiconTrieStore(
        words=marisa_trie.Trie(words or []),
        entity_phrases=marisa_trie.Trie(phrases),
        entity_tokens=marisa_trie.Trie(entity_tokens or []),
        metadata={
            "max_entity_phrase_tokens": max(
                (len(phrase.split(" ")) for phrase in phrases),
                default=1,
            )
        },
    )


def ged_input_from_preprocessing(
    text: str,
    *,
    use_morphology: bool = True,
    show_preprocessing: bool = False,
) -> GEDInput:
    """Build GED input using the preprocessing service."""
    pre_output = preprocess(PreprocessingInput(text=text))

    if show_preprocessing:
        print("Preprocessing output:")
        print(json.dumps(pre_output.model_dump(), ensure_ascii=False, indent=2))

    morph_features = (
        pre_output.morph_features if use_morphology else [[] for _ in pre_output.tokens]
    )

    return GEDInput(
        text=pre_output.text,
        normalized_text=pre_output.normalized_text,
        tokens=pre_output.tokens,
        morph_features=morph_features,
    )


def show_errors(text: str, errors) -> None:
    """Print GED errors in the same compact style as the rule-based notebook."""
    if not errors:
        print("No errors found")
        return

    print(f"Found {len(errors)} errors:")
    for i, error in enumerate(errors, start=1):
        print("*" * 20 + f" Error {i} " + "*" * 20)
        print(f"  Error in: {text[error.span[0] : error.span[1]]}")
        print(f"  Error type: {error.category}-{error.subtype}")
        print(f"  Sources: {[source.value for source in error.sources]}")
        print(f"  Tier: {error.provenance_tier}")
        print(f"  Confidence: {error.confidence}")
        print(f"  Error description: {error.explanation_text}")


def test_detector(
    detector,
    text: str,
    *,
    use_morphology: bool = True,
    show_preprocessing: bool = False,
    sum_output: bool = True,
):
    """Run one detector after preprocessing."""
    payload = ged_input_from_preprocessing(
        text,
        use_morphology=use_morphology,
        show_preprocessing=show_preprocessing,
    )
    errors = detector.detect(
        payload.text,
        payload.normalized_text,
        payload.tokens,
        payload.morph_features,
    )

    if sum_output:
        show_errors(payload.text, errors)
    else:
        print(
            json.dumps(
                [error.model_dump() for error in errors], ensure_ascii=False, indent=2
            )
        )


def test_service(
    service: GEDService,
    text: str,
    *,
    use_morphology: bool = True,
    show_preprocessing: bool = False,
    sum_output: bool = True,
):
    """Run GED service after preprocessing."""
    payload = ged_input_from_preprocessing(
        text,
        use_morphology=use_morphology,
        show_preprocessing=show_preprocessing,
    )
    output = service.process(payload)

    if sum_output:
        show_errors(output.text, output.errors)
    else:
        print(json.dumps(output.model_dump(), ensure_ascii=False, indent=2))

    return payload, output

---

## Pattern Matching



In [4]:
pattern_detector = LexiconDetector(enable_spelling_suspicion=False)

print(f"Loaded {len(pattern_detector.list_patterns())} curated lexicon patterns")

2026-06-15 21:34:02.440 | INFO     | src.services.ged.detectors.lexicon.loader:load_patterns:49 - Loaded 2 lexicon patterns from /home/amir/dev/baligh/src/services/ged/detectors/lexicon/resources/patterns.yaml.


Loaded 2 curated lexicon patterns


In [ ]:
test_detector(pattern_detector, "لا كن هذا مؤلم.")

[2026-06-15 21:34:02,692 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


In [ ]:
# merge pattern: one typed token should be multiple tokens
test_detector(pattern_detector, "انشاءالله خير")

Found 1 errors:
******************** Error 1 ********************
  Error in: انشاءالله
  Error type: MG-common_merge
  Sources: ['lexicon_matcher']
  Tier: tier_1_rule_derived
  Confidence: 1.0
  Error description: الصواب فصل العبارة: إن شاء الله.


---

## Dictionary-backed Spelling Suspicion



In [ ]:
tiny_store = make_store(
    words=["مدرسه", "كتاب", "البيت"],
    entity_phrases=["حي اول المحله"],
    entity_tokens=["احمد"],
)

spelling_detector = LexiconDetector(
    patterns=[],
    trie_store=tiny_store,
    enable_spelling_suspicion=True,
)

In [ ]:
test_detector(spelling_detector, "كتاب.", use_morphology=True)

No errors found


In [ ]:
test_detector(spelling_detector, "مدرسة.", use_morphology=False)

No errors found


In [ ]:
test_detector(spelling_detector, "قلم.", use_morphology=False)

Found 1 errors:
******************** Error 1 ********************
  Error in: قلم
  Error type: OT-spelling
  Sources: ['lexicon_matcher']
  Tier: tier_2_rule_supported
  Confidence: 0.8
  Error description: الكلمة غير موجودة في المعجم المتاح ولم يقدم المحلل الصرفي تحليلا موثوقا.


In [ ]:
test_detector(spelling_detector, "حي أول المحلة.", use_morphology=False)

No errors found


In [ ]:
test_detector(spelling_detector, "أحمد.", use_morphology=False)

No errors found


---

## GED Service with Rule-based + Lexicon Matcher



In [ ]:
combined_service = GEDService(
    subsystems=[
        RuleBasedDetector(),
        LexiconDetector(enable_spelling_suspicion=False),
    ]
)

2026-06-15 21:33:56.301 | INFO     | src.services.ged.detectors.lexicon.loader:load_patterns:49 - Loaded 2 lexicon patterns from /home/amir/dev/baligh/src/services/ged/detectors/lexicon/resources/patterns.yaml.


In [ ]:
# Rule-based + lexicon in one GED service call.
payload, output = test_service(
    combined_service,
    "ذهب الى البيت ، لا كن انشاءالله سيعود.",
    show_preprocessing=False,
)

Found 4 errors:
******************** Error 1 ********************
  Error in: الى
  Error type: OT-hamza
  Sources: ['rule_based']
  Tier: tier_1_rule_derived
  Confidence: 1.0
  Error description: حرف الجر أو الربط يبدأ بهمزة قطع (إ/أ) لا بألف مجردة (ا)؛ مثل: إلى، إن، أن , لا: الى، ان، ان
******************** Error 2 ********************
  Error in: ،
  Error type: PC-spacing
  Sources: ['rule_based']
  Tier: tier_1_rule_derived
  Confidence: 1.0
  Error description: علامة الترقيم يجب أن تلتصق بالكلمة التي تسبقها دون فراغ؛ مثل: «ذهب، ثم» لا «ذهب ، ثم»
******************** Error 3 ********************
  Error in: لا كن
  Error type: SP-common_split
  Sources: ['lexicon_matcher']
  Tier: tier_1_rule_derived
  Confidence: 1.0
  Error description: تكتب الكلمة متصلة: لكن.
******************** Error 4 ********************
  Error in: انشاءالله
  Error type: MG-common_merge
  Sources: ['lexicon_matcher']
  Tier: tier_1_rule_derived
  Confidence: 1.0
  Error description: الصواب فصل العبارة: إ

In [ ]:
print(json.dumps(output.model_dump(), ensure_ascii=False, indent=2))

{
  "text": "ذهب الى البيت ، لا كن انشاءالله سيعود.",
  "errors": [
    {
      "span": [
        4,
        7
      ],
      "token_refs": [
        1
      ],
      "category": "OT",
      "subtype": "hamza",
      "confidence": 1.0,
      "sources": [
        "rule_based"
      ],
      "provenance_tier": "tier_1_rule_derived",
      "explanation_eligible": true,
      "explanation_text": "حرف الجر أو الربط يبدأ بهمزة قطع (إ/أ) لا بألف مجردة (ا)؛ مثل: إلى، إن، أن , لا: الى، ان، ان"
    },
    {
      "span": [
        14,
        15
      ],
      "token_refs": [
        3
      ],
      "category": "PC",
      "subtype": "spacing",
      "confidence": 1.0,
      "sources": [
        "rule_based"
      ],
      "provenance_tier": "tier_1_rule_derived",
      "explanation_eligible": true,
      "explanation_text": "علامة الترقيم يجب أن تلتصق بالكلمة التي تسبقها دون فراغ؛ مثل: «ذهب، ثم» لا «ذهب ، ثم»"
    },
    {
      "span": [
        16,
        21
      ],
      "token_refs": [
 